In [ ]:
from sys import platform
from subprocess import run
from time import sleep
from random import randint

# Variaveis Globais
ASCII = 64
barcos =["varzea", "corveta", "submarino", "cruzador", "torpedeiro", "porta aviões"]
barcosTamanho = ["varzea", 2, 3, 3, 4, 5]
barcosAcertos = [["varzea", 0, 0, 0, 0, 0], ["varzea", 0, 0, 0, 0, 0]]

# Variaveis usadas no salvamento
saveGameMaps = []
saveGameAcertos = []
saveTurnState = False

# Variaveis de lógica de jogo
placed = False
turno = False
victory = [False, False]
cor_ativada = True

# Códigos de Cor
AGUA  = "\033[34m"
BARCO = "\033[32m"
TIRO  = "\033[31m"
RESET = "\033[0m"

# Muda cor do mapa se cor for ativada nas configurações
def cor(codigo):
    return codigo if cor_ativada else ""

# Limpa a tela e pausa
def clearScreen(time = .5):
    if platform == "win32":
        sleep(time)
        run("cls", shell=True)

    else:
        sleep(time)
        run(["clear"])

# Pede por coordenada e traduz de letra pra ascii, tambem verifica se coordenada inserida
def translate_coord():
    while True:
        coord = input("Digite a coordenada:\n Ex: A1 \n").upper()
        if len(coord) == 3 or len(coord) == 2:

            x = ord(coord[0]) - 64

            if len(coord) > 2:
                y = int(coord[1] + coord[2])

            else:
                y = int(coord[1])

            if x > size or y > size:
                print("Posição Inválida\n")

            else:
                break
        else:
            print("Posição Inválida\n")

    return [x, y]

# Função pra verificar se já existe um barco onde o jogador esta tentando posicionar
def boat_check(matriz, xis, ypsilon, direc, tamanho):
    if direc == "h":
        if ypsilon + tamanho > len(matriz):
            return True

        for i in range(ypsilon, ypsilon + tamanho):
            if matriz[xis][i] in [1, 2, 3, 4, 5]:
                return True

    elif direc == "v":
        if xis + tamanho > len(matriz):
            return True

        for i in range(xis, xis + tamanho):
            if matriz[i][ypsilon] in [1, 2, 3, 4, 5]:
                return True

    else:
        return False

# Posiciona o barco
def place_boat(matriz, barco, ordem, coord):

    chi = coord[0]
    upsilon = coord[1]

    tamanho = barcosTamanho[barcos.index(barco)]

    matrizAtualizada = matriz

    if ordem == "h":
        for i in range(upsilon, upsilon + tamanho):
            matrizAtualizada[chi][i] = barcos.index(barco)
    elif ordem == "v":
        for i in range(chi, chi + tamanho):
            matrizAtualizada[i][upsilon] = barcos.index(barco)

    return matrizAtualizada

# Atira e passa o turno
def shoot_boat(matriz, turn):
    global turno

    list_coord = translate_coord()

    xray = list_coord[0]
    yankee = list_coord[1]

    if matriz[int(not turn)][xray][yankee] == 0:
        matriz[int(not turn)][xray][yankee] = 6
        turno = not turn
        print("Errou!")

    else:
        for i in range(1, len(barcos)):
            if matriz[int(not turn)][xray][yankee] == i:
                matriz[int(not turn)][xray][yankee] = 6
                barcosAcertos[int(turn)][i] += 1

                if barcosAcertos[int(turn)][i] == barcosTamanho[i]:
                    print(f"\nDestruiu {barcos[i]}\n")

                else:
                    print(f"\nAcertou {barcos[i]}\n")

        if barcosAcertos[int(turn)] != barcosTamanho:
            print("Jogue Novamente!")

    if barcosAcertos[int(turno)] == barcosTamanho:
        victory[int(turn)] = True

# Gera a interface grafica do mapa
def print_map(matriz):
    global placed

    print_list = ["~~", "<<", "°°", "©©", "^^", "[]", "XX"]

    for i in range(len(matriz)):
        line = ""

        for k in range(len(matriz[i])):
            if i == 0 and 0 < k:
                if k < 10:
                    line += ("0" + str(matriz[i][k]))
                else:
                    line += (str(matriz[i][k]))

            elif k == 0:
                line += str(matriz[i][k])

            elif i > 0 and placed == False:
                val = matriz[i][k]
                if val == 0:
                    line += cor(AGUA) + print_list[0] + cor(RESET)
                elif val == 6:
                    line += cor(TIRO) + print_list[6] + cor(RESET)
                else:
                    line += cor(BARCO) + print_list[val] + cor(RESET)

            else:
                val = matriz[i][k]
                if val == 6:
                    line += cor(TIRO) + print_list[6] + cor(RESET)
                else:
                    line += cor(AGUA) + print_list[0] + cor(RESET)

            line += " "
        print(line)

# Executa a logica de posicionamento de barcos
def placement(matriz):

    for i in range(len(barcos) -1, 0, -1):
        matrizNova = matriz
        while True:
            clearScreen()
            print_map(matrizNova)

            print("\n")
            print(f"Posicionando o {barcos[i]}")
            list_coordenada = translate_coord()

            𒉽 = list_coordenada[0]
            𒌨 = list_coordenada[1]

            dirc = input(f"Digite 'V' para posicionar verticalmente ou digite 'H' para posicionar horizontalmente \n" ).lower().strip()

            if dirc == "h" and (𒉽 >= len(matriz) or 𒌨 + barcosTamanho[i] > len(matriz)):
                print("Posição inválida horizontalmente (saiu do mapa), escolha novamente\n")

            elif dirc == "v" and (𒉽 + barcosTamanho[i] > len(matriz) or 𒌨 >= len(matriz)):
                print("Posição inválida verticalmente (saiu do mapa), escolha novamente\n")

            # A condição "or" so precisava que um retornasse verdadeiro, então como "v" e "h" são sempre diferente um do outro sempre
            # Entrava no elif e dava erro.
            elif dirc != "v" and dirc != "h":
                print("Direção invalida \n")

            elif boat_check(matriz, 𒉽, 𒌨, dirc, barcosTamanho[i]):
                print("Posição já esta ocupada \n")

            else:
                matrizNova = place_boat(matriz, barcos[i], dirc, list_coordenada)
                print(f"{barcos[i]} posicionado \n")
                break

# Cria o mapa baseado no tamanho
def create_map(tamanho):
    matriz = []

    for i in range(tamanho + 1):
        row = []

        for k in range(tamanho + 1):

            if k == 0 and i == 0:
                row.append(chr(ASCII))

            elif i == 0:
                row.append(k)

            elif k == 0:
                row.append(chr(ASCII + i))

            else:
                row.append(0)

        matriz.append(row)

    return matriz

# Gera o menu e retorna a escolha
def menu():
    bt = "BATALHA NAVAL"
    opt = ["1. Começar a Jogar", "2. Jogo Rápido", "3. Configurações", "4. Carregar Jogo Salvo","5. Deletar Jogo Salvo", "6. Sair do jogo"]
    print(f"{bt:=^27}\n")

    for i in range(len(opt)):
        print(f"{opt[i]:^{27}}")

    while True:
        choice = int(input("\nSelecione uma Opção:\n"))

        if choice > len(opt) or choice < 1:
            print(f"{'Opção Invalida':^27}\n")

        else:
            return choice

# Gera o menu de configurações
def configuracoes():
    global cor_ativada

    while True:
        clearScreen()
        status = f"{cor(AGUA)}LIGADAS{cor(RESET)}" if cor_ativada else "DESLIGADAS"
        print("=" * 27)
        print(f"{'CONFIGURAÇÕES':^27}")
        print("=" * 27)
        print(f"\n  Cores do terminal: {status}\n")
        print("  1. Ligar cores")
        print("  2. Desligar cores")
        print("  3. Voltar ao menu\n")

        escolha = input("Selecione uma opção: ").strip()

        if escolha == "1":
            cor_ativada = True
            print("\nCores ativadas!\n")
            clearScreen()

        elif escolha == "2":
            cor_ativada = False
            print("\nCores desativadas!\n")
            clearScreen()

        elif escolha == "3":
            break
        else:
            print("\nOpção inválida!\n")
            clearScreen()

# Começa o jogo posicionando os barcos de ambos os jogadores
def start_game():
    global turno
    global placed
    placed = False
    for i in range(2):
        print(f"Jogador {int(turno) + 1} posicione seus barcos\n")

        placement(players[int(turno)])

        print_map(players[int(turno)])
        turno = not turno

# Executa a logica de jogo com menu
def jogar():
    global saveGameMaps
    global saveGameAcertos
    global saveTurnState
    global placed

    placed = True
    opt = 0

    while True not in victory:

        clearScreen(1)

        print(f"\n Turno do jogador {int(turno) + 1}\n")

        while opt > 4 or opt < 1:
            print("\n1. Atacar")
            print("2. Sair e Salvar")
            print("3. Sair sem Salvar")
            print("4. Desistir\n")

            opt = int(input("Selecione uma opção: "))

        match opt:
            case 1:
                print_map(players[int(not turno)])
                print("\n")
                shoot_boat(players, turno)
                opt = 0

            case 2:
                saveGameMaps = players.copy()
                saveGameAcertos = barcosAcertos.copy()
                saveTurnState = turno
                print("\n Jogo Salvo com Sucesso!")
                clearScreen()
                break

            case 3:
                clearScreen()
                break

            case 4:
                print(f"O jogador {int(not turno) + 1} ganhou!\n")
                break

# Posiciona barcos aleatóriamente pelo mapa, garantindo que estão em posições corretas antes de posicionar
def randomPlacement(mapSize):
    global placed
    placed = False

    dirs = ["v", "h"]

    results = []

    for i in range(2):
        matriz = create_map(mapSize)

        for j in range(len(barcos) - 1, 0, -1):
            tamanho = barcosTamanho[j]

            while True:
                dirc = dirs[randint(0, 1)]

                if dirc == "h":
                    row = randint(1, mapSize)
                    col = randint(1, mapSize - tamanho + 1)
                else:
                    row = randint(1, mapSize - tamanho + 1)
                    col = randint(1, mapSize)

                coord = [row, col]

                if not boat_check(matriz, row, col, dirc, tamanho):
                    matriz = place_boat(matriz, barcos[j], dirc, coord)
                    break

        results.append(matriz)

    return results

# Executa o jogo
while True:
    clearScreen()

    option = menu()
    placed = False

    match option:
        # Jogo Normal
        case 1:
            clearScreen()

            print(f"Bem vindo à Batalha Naval!\n")

            size = int(input("Digite o tamanho do mapa: \n"))

            while size < 10 or size > 20:
                clearScreen()
                print("O mapa não pode ter o tamanho menor que 10x10 ou maior do que 20x20.")
                size = int(input("Digite o tamanho do mapa: \n"))

            playerOneMatriz = create_map(size)
            playerTwoMatriz = create_map(size)
            players = [playerOneMatriz, playerTwoMatriz]

            start_game()

            clearScreen(0.7)

            jogar()

        # Jogo Rapido
        case 2:
            clearScreen()

            print(f"Bem vindo à Batalha Naval!\n")

            size = int(input("Digite o tamanho do mapa: \n"))

            while size < 10 or size > 20:
                clearScreen()
                print("O mapa não pode ter o tamanho menor que 10x10 ou maior do que 20x20.")
                size = int(input("Digite o tamanho do mapa: \n"))

            playerOneMatriz = randomPlacement(size)[0]
            playerTwoMatriz = randomPlacement(size)[0]
            players = [playerOneMatriz, playerTwoMatriz]

            jogar()

        # Configurações
        case 3:
            clearScreen(0.7)
            configuracoes()

            clearScreen()

        # Carregar jogo salvo
        case 4:
            if len(saveGameMaps) == 0:
                print("Nenhum jogo salvo foi encontrado\n")

            else:
                players = saveGameMaps.copy()
                barcosAcertos = saveGameAcertos.copy()
                turno = saveTurnState
                print("Jogo Carregado\n")

                clearScreen()

                jogar()
            clearScreen()

        # Deletar jogo salvo
        case 5:
            saveTurnState = False
            saveGameAcertos = []
            saveGameMaps = []
            print("Jogo salvo foi deletado")
            clearScreen()

        # Fechar jogo
        case 6:
            exit()

    # Verifica vitoria e mostra o tabuleiro de ambos os jogadores com placed = False
    if barcosAcertos[int(turno)] == barcosTamanho:
        print(f"O jogador {int(turno) + 1} ganhou!\n")
        placed = False
        print("Tabuleiro do Jogador 1:\n")
        print_map(players[0])

        print("Tabuleiro do Jogador 2:\n")
        print_map(players[1])